# Scraping des Restaurants de Marrakech – TripAdvisor
**Partie A :** Collecter top 500 restaurants

## Installation

In [7]:
!pip install requests beautifulsoup4 fake_useragent pandas


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports & Configuration

In [1]:
import requests
from bs4 import BeautifulSoup
from fake_useragent import UserAgent
import json, time, random, re
import pandas as pd
import os

BASE_URL    = 'https://www.tripadvisor.com'
LISTING_URL = 'https://www.tripadvisor.com/Restaurants-g293734-Marrakech_Marrakech_Tensift_El_Haouz_Region.html'
PAGE_SIZE   = 30

ua = UserAgent()
print('Configuration chargée')

Configuration chargée


##  Anti-blocage : NE S'ARRÊTE pas sur 403

In [2]:
def get_headers():
    return {
        'User-Agent'      : ua.random,
        'Accept-Language' : random.choice(['fr-FR,fr;q=0.9', 'en-US,en;q=0.9', 'ar-MA,ar;q=0.8']),
        'Referer'         : 'https://www.google.com/',
        'DNT'             : '1',
        'Connection'      : 'keep-alive',
        'Accept'          : 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    }

def safe_request(url):
    """
    Tente indéfiniment jusqu'à obtenir une réponse 200.
    Sur 403/429 : pause longue progressive puis réessai.
    Ne retourne jamais None — continue jusqu'au succès.
    """
    attempt = 0
    while True:
        attempt += 1
        time.sleep(random.uniform(2, 4))
        try:
            resp = requests.get(url, headers=get_headers(), timeout=15)

            if resp.status_code == 200:
                return resp

            elif resp.status_code in [403, 429]:
                # Pause progressive : 60s, 120s, 180s ... max 300s
                wait = min(60 * attempt, 300) + random.uniform(0, 30)
                
                time.sleep(wait)
                # Réinitialise le compteur après une longue pause
                if attempt >= 3:
                    attempt = 0

            else:
                wait = random.uniform(5, 15)
                print(f'  HTTP {resp.status_code} – attente {wait:.0f}s')
                time.sleep(wait)

        except Exception as e:
            print(f'Erreur réseau : {e} – réessai dans 10s')
            time.sleep(10)

print('safe_request définie ')

safe_request définie 


##  URL paginée & total réel

In [3]:
def build_page_url(offset):
    if offset == 0:
        return LISTING_URL
    return LISTING_URL.replace(
        'Restaurants-g293734-',
        f'Restaurants-g293734-oa{offset}-'
    )

def get_total_from_page(soup):
    for tag in soup.find_all(['span', 'div', 'h1', 'b']):
        text = tag.get_text(strip=True)
        m = re.search(r'^([\d\s,]+)\s+restaurant', text, re.IGNORECASE)
        if m:
            total = int(re.sub(r'[^\d]', '', m.group(1)))
            if 100 < total < 10000:
                return total
    return None

print('Fonctions URL définies')

Fonctions URL définies


## Parser les cartes restaurants

In [4]:
def extract_restaurants_from_page(soup):
    found = []
    cards = soup.find_all('div', class_='ZvrsW N G')
    for card in cards:
        try:
            link_el = card.find('a', class_='BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS')
            if not link_el:
                continue
            name = link_el.text.strip()
            link = BASE_URL + link_el['href'].strip()

            rate = None
            title_el = card.find('title')
            if title_el:
                m = re.search(r'([\d\.]+)', title_el.text)
                if m:
                    rate = float(m.group(1))

            nb_reviews = None
            for span in card.find_all('span'):
                m = re.search(r'([\d,\s]+)\s*(avis|review)', span.get_text(), re.IGNORECASE)
                if m:
                    nb_reviews = int(re.sub(r'[^\d]', '', m.group(1)))
                    break

            found.append({'name': name, 'link': link, 'rate': rate, 'nb_reviews': nb_reviews})
        except (KeyError, AttributeError, TypeError):
            continue
    return found

print('Parser défini')

Parser défini


## Scraping principal


In [5]:
# ── Charger checkpoint si existant ────────────────────────────────────
CHECKPOINT_FILE = 'data/restaurants_marrakech_partial.json'

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
        saved = json.load(f)
    all_restaurants = {v['link']: v for v in saved.values()}
else:
    all_restaurants = {}

offset          = 0
page            = 1
total_available = None

while True:
    url  = build_page_url(offset)
    resp = safe_request(url)
    soup = BeautifulSoup(resp.text, 'html.parser')

    if page == 1:
        total_available = get_total_from_page(soup)

    batch = extract_restaurants_from_page(soup)

    new_count = 0
    for r in batch:
        if r['link'] not in all_restaurants:
            all_restaurants[r['link']] = {
                'name'      : r['name'],
                'rate'      : r['rate'],
                'nb_reviews': r['nb_reviews'],
                'link'      : r['link']
            }
            new_count += 1

    if page % 10 == 0:
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
            json.dump({v['name']: v for v in all_restaurants.values()}, f, ensure_ascii=False, indent=4)

    if new_count == 0:
        break

    offset += PAGE_SIZE
    page   += 1

    if total_available and offset >= total_available:
        break

with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
    json.dump({v['name']: v for v in all_restaurants.values()}, f, ensure_ascii=False, indent=4)

## Sauvegarde finale

In [6]:

restaurants_by_name = {
    v['name']: {'link': v['link'], 'rate': v['rate'], 'nb_reviews': v['nb_reviews']}
    for v in all_restaurants.values()
}
with open('data/restaurants_marrakech.json', 'w', encoding='utf-8') as f:
    json.dump(restaurants_by_name, f, ensure_ascii=False, indent=4)
print(f'data/restaurants_marrakech.json — {len(restaurants_by_name)} restaurants')

data/restaurants_marrakech.json — 1781 restaurants


## Top 500 + export CSV

In [7]:
with open('data/restaurants_marrakech.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f'Total chargé : {len(data)} restaurants')

top_500 = sorted(
    data.items(),
    key=lambda x: x[1]['rate'] if x[1]['rate'] is not None else 0,
    reverse=True
)[:500]

top_500_dict = {name: info for name, info in top_500}

with open('data/top_500_restaurants.json', 'w', encoding='utf-8') as f:
    json.dump(top_500_dict, f, ensure_ascii=False, indent=4)

df = pd.DataFrame([
    {'rank': i+1, 'name': name, 'rate': info['rate'],
     'nb_reviews': info['nb_reviews'], 'link': info['link']}
    for i, (name, info) in enumerate(top_500_dict.items())
])
df.to_csv('data/top_500_restaurants.csv', index=False, encoding='utf-8-sig')

print(f'Top 500 sauvegardé')
print(f'   1er  : {top_500[0][0]}  (note={top_500[0][1]["rate"]})')
print(f'   500e : {top_500[-1][0]} (note={top_500[-1][1]["rate"]})')
display(df[['rank','name','rate','nb_reviews']].head(10))


Total chargé : 1781 restaurants
Top 500 sauvegardé
   1er  : 25. Kach Kech Rooftop  (note=5.0)
   500e : 200. Rivayat (note=4.7)


,rank,name,rate,nb_reviews
0,1,25. Kach Kech Rooftop,5.0,131
1,2,112. Zola marrakech,5.0,89
2,3,175. Krepchy,5.0,70
3,4,291. Burger Les Dalton Marrakech,5.0,41
4,5,303. Kissa Rooftop,5.0,12
5,6,316. La Fantasia Italian Pizza & Food,5.0,35
6,7,335. Maison KA Restaurant,5.0,42
7,8,"356. Le Travertin, Batbout Cheese Steak",5.0,31
8,9,363. Myogi Asian Restaurant,5.0,11
9,10,367. Sky Bar Wow,5.0,9
